In [12]:
#this code with reload any imports if you're updating them actively
%reload_ext autoreload
%autoreload 2
import pygem.pygem_input as pygem_prms
import scipy.optimize as opt
import numpy as np
import xarray as xr
import pandas as pd
import os
import suncalc as solar
#PyGEM
import pygem.pygem_input as pygem_prms
import pygem.oggm_compat as oggm
import pygem.pygem_modelsetup as modelsetup
import class_climate

In [2]:
# # read 10 year data files for each variable in varnames and merge
# eb_varnames = ['temp','dtemp','precip','surfrad','tcc','uwind','vwind']
# i=2
# varname = 'press'
# hourlyfp = '~/research/climate_data/ERA5/ERA5_hourly/varname/ERA5_varname_hourly'.replace('varname',varname)
# da_0 = xr.open_dataarray(hourlyfp + '_80_89.nc')
# da_1 = xr.open_dataarray(hourlyfp + '_90_99.nc')
# da_2 = xr.open_dataarray(hourlyfp + '_00_09.nc')
# da_3 = xr.open_dataarray(hourlyfp + '_10_21.nc')
# print(da_0.coords)
# print(da_1.coords)
# print(da_2.coords)
# print(da_3.coords)

# da = xr.merge([da_0,da_1,da_2,da_3])
# da.to_netcdf('~/research/climate_data/ERA5/ERA5_hourly/ERA5_varname_hourly.nc'.replace('varname',varname))
# print(da)
#368,164

In [13]:
if pygem_prms.glac_no not in ['01.00570'] and pygem_prms.run_eb:
    print('EB model can currently only run Gulkana glacier')
glacier_table = modelsetup.selectglaciersrgitable(pygem_prms.glac_no,
                rgi_regionsO1=pygem_prms.rgi_regionsO1, rgi_regionsO2=pygem_prms.rgi_regionsO2,
                rgi_glac_number=pygem_prms.rgi_glac_number, include_landterm=pygem_prms.include_landterm,
                include_laketerm=pygem_prms.include_laketerm, include_tidewater=pygem_prms.include_tidewater)
# ===== TIME PERIOD =====
dates_table = modelsetup.datesmodelrun(startyear=pygem_prms.gcm_startyear,endyear=pygem_prms.gcm_endyear, 
                                       spinupyears=pygem_prms.gcm_spinupyears,option_wateryear=pygem_prms.gcm_wateryear)

EB model can currently only run Gulkana glacier
1 glaciers in region 1 are included in this model run: ['00570']
This study is focusing on 1 glaciers in region [1]
!! Not handling water years in modelsetup.datesmodelrun


In [4]:
#%% MODEL PROPERTIES
density_ice = 900           # Density of ice [kg m-3] (or Gt / 1000 km3)
density_water = 1000        # Density of water [kg m-3]
area_ocean = 362.5 * 1e12   # Area of ocean [m2] (Cogley, 2012 from Marzeion et al. 2020)
k_ice = 2.33                # Thermal conductivity of ice [J s-1 K-1 m-1] recall (W = J s-1)
k_air = 0.023               # Thermal conductivity of air [J s-1 K-1 m-1] (Mellor, 1997)
k_air = 0.001               # Thermal conductivity of air [J s-1 K-1 m-1]
ch_ice = 1890000            # Volumetric heat capacity of ice [J K-1 m-3] (density=900, heat_capacity=2100 J K-1 kg-1)
ch_air = 1297               # Volumetric Heat capacity of air [J K-1 m-3] (density=1.29, heat_capacity=1005 J K-1 kg-1)
Lh_rf = 333550              # Latent heat of fusion [J kg-1]
tolerance = 1e-12           # Model tolerance (used to remove low values caused by rounding errors)
gravity = 9.81              # Gravity [m s-2]
pressure_std = 101325       # Standard pressure [Pa]
temp_std = 288.15           # Standard temperature [K]
R_gas = 8.3144598           # Universal gas constant [J mol-1 K-1]
molarmass_air = 0.0289644   # Molar mass of Earth's air [kg mol-1]


# MORE FACTORS THAT MIGHT BE NECESSARY
kp = 1                              # precipitation factor [-] (referred to as k_p in Radic etal 2013; c_prec in HH2015)
tbias = 5                           # temperature bias [deg C]
ddfsnow = 0.0041                    # degree-day factor of snow [m w.e. d-1 degC-1]
ddfsnow_iceratio = 0.7              # Ratio degree-day factor snow snow to ice
ddfice = ddfsnow / ddfsnow_iceratio # degree-day factor of ice [m w.e. d-1 degC-1]
precgrad = 0.0001                   # precipitation gradient on glacier [m-1]
lapserate = -0.0065                 # temperature lapse rate for both gcm to glacier and on glacier between elevation bins [K m-1]
tsnow_threshold = 1                 # temperature threshold for snow [deg C] (HH2015 used 1.5 degC +/- 1 degC)
calving_k = 0.7                     # frontal ablation rate [yr-1]

In [5]:
# %load_ext line_profiler
# %reload_ext autoreload
# %autoreload 2
# import class_climate
# gcm = class_climate.GCM(name='ERA5-hourly')
# %lprun -f gcm.importGCMvarnearestneighbor_xarray gcm.importGCMvarnearestneighbor_xarray(gcm.vwind_fn, gcm.vwind_vn, glacier_table,dates_table)

In [16]:
# ===== LOAD CLIMATE DATA =====
gcm = class_climate.GCM(name='ERA5-hourly')
gcm_prec, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.prec_fn, gcm.prec_vn, glacier_table,dates_table)
gcm_temp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.temp_fn, gcm.temp_vn, glacier_table,dates_table)
gcm_dtemp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.dtemp_fn, gcm.dtemp_vn, glacier_table,dates_table)
gcm_sp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.press_fn, gcm.press_vn, glacier_table,dates_table)
gcm_tcc, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.tcc_fn, gcm.tcc_vn, glacier_table,dates_table)
gcm_surfrad, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.surfrad_fn, gcm.surfrad_vn, glacier_table,dates_table) 
gcm_uwind, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.uwind_fn, gcm.uwind_vn, glacier_table,dates_table)                                                      
gcm_vwind, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.vwind_fn, gcm.vwind_vn, glacier_table,dates_table)

gcm_elev = gcm.importGCMfxnearestneighbor_xarray(gcm.elev_fn, gcm.elev_vn, glacier_table)

!! Not checking units for any EB variables
!! Not checking units for any EB variables
!! Not checking units for any EB variables
!! Not checking units for any EB variables
!! Not checking units for any EB variables
!! Not checking units for any EB variables


In [7]:
# READ GULKANA ELEVATION DATA FROM RGI 6.0
gulkana = '01.00570'
glac_no = [gulkana]

# ----- RGI DATA -----
# Filepath for RGI files
main_directory = os.getcwd()
elev_points = glacier_table[['Zmin','Zmed','Zmax']]

In [8]:
# READ GULKANA ELEVATIONS FROM OGGM GDIRS
#this step is specific to the EB for three points on Gulkana
#to generalize, need a variable geometry containing zwh for the points for the EB
gdir = oggm.single_flowline_glacier_directory(glac_no[0], logging_level='CRITICAL')
fls = oggm.get_glacier_zwh(gdir)
#filter out zero bins to get only initial glacier volume
fls = fls.iloc[np.nonzero(fls['h'].to_numpy())]
z_stats = np.array([np.min(fls['z']),np.median(fls['z']),np.max(fls['z'])])

print('Min, median and max bin elevations from OGGM flowlines:')
print(pd.Series(z_stats,index=['Zmin','Zmed','Zmax']))
print('Min, median and max elevation from RGI:')
print(elev_points)

#setup three points at minimum, median and maximum elevation band from OGGM
median_index = np.where(fls['z']==z_stats[1])[0][0]
w_stats = np.array([fls['w'][len(fls)-1],fls['w'][median_index],fls['w'][0]])
h_stats = np.array([fls['h'][len(fls)-1],fls['h'][median_index],fls['h'][0]])
geo_index = ['Bottom','Middle','Top']
geometry = pd.DataFrame({'z':z_stats,'w':w_stats,'h':h_stats},index=geo_index)

Min, median and max bin elevations from OGGM flowlines:
Zmin    1172.809436
Zmed    1628.401286
Zmax    2330.675374
dtype: float64
Min, median and max elevation from RGI:
   Zmin  Zmed  Zmax
0  1162  1858  2438


In [9]:
# ### this whole block is probably useless

# #manually set number of exponentially scaling bins
# n_vert_bins = 10
# n_points = len(geo_index)
# option_bin = 0

# #create variable to store glacier geometry
# vert_bins = xr.Dataset(data_vars = dict(
#     bin_w = (['pt'],geometry['w']),
#     bin_z = (['pt'],geometry['z'])),
#     coords=dict(
#         pt=(['pt'],geo_index),
#         vert_idx=range(n_vert_bins)
#         )
#     )

# bin_depths = np.zeros((2,n_points,n_vert_bins))
# #fill vertical bin heights based on ice thickness
# for g in range(n_points):
#     #get ice thickness of current point
#     pt_h = geometry['h'].iloc[g]
    
#     if option_bin==0:
#         hs = [0.1,.25,.5,.75,1,2,5,10,20,pt_h-39.6]
#         ds = [sum(hs[:i]) for i in range(n_vert_bins)]
#         bin_depths[:,g,:] = [hs,ds]
#     else:
#         c = opt.fsolve(lambda c: pt_h-np.sum(np.exp(np.arange(n_vert_bins)*c)),10)
#         bin_depths[g,:] = np.exp(c*range(1,n_vert_bins))
# vert_bins['bin_h'] = (['pt','vert_idx'],bin_depths[0,:,:])
# vert_bins['bin_d'] = (['pt','vert_idx'],bin_depths[1,:,:])

# #set bin content as a constant snow, firn or ice (s,f,i)
# content_arr = np.empty((n_points,n_vert_bins),dtype=str)
# content_arr[0,:] = ['i']*n_vert_bins
# content_arr[1,:] = ['f']*round(n_vert_bins*.3)+['i']*round(n_vert_bins*.7)
# content_arr[2,:] = ['s']*round(n_vert_bins*.2)+['f']*round(n_vert_bins*.2)+['i']*round(n_vert_bins*.6)
# vert_bins['bin_content'] = (['pt','vert_idx'],content_arr)

# #since using constant bin content, can also use constant properties of snow/ice
# vert_bins['lambdas'] = (['pt','vert_idx'],np.where(content_arr=='s',0.9,.8))
# vert_bins['rs'] = (['pt','vert_idx'],np.where(content_arr=='s',17.1,2.5))

In [17]:
# ===== SET UP CLIMATE DATASETS =====
#create dataset to store variables that need to be downscaled by elevation
#if we want to be able to run multiple glaciers at once, this will need to be updated to add 'glacier' as a coordinate
climateds_binned = xr.Dataset(data_vars = dict(
    bin_elev = (vert_bins['bin_z'])),
    coords=dict(
        pt=(['pt'],geo_index),
        time=gcm_hours
        ),
    attrs=dict(description="Climate data adjusted for points in EB."))

#initialize variables to be adjusted
temp_adj = np.zeros((n_points,len(gcm_hours)))
prec_adj = np.zeros((n_points,len(gcm_hours)))
sp_adj = np.zeros((n_points,len(gcm_hours)))
rh_adj = np.zeros((n_points,len(gcm_hours)))

# define function to calculate RH from temp and dewpoint temp
e_func = lambda T: 6.1078*np.exp(17.1*T/(235+T)) #vapor pressure in hPa

#loop through each elevation bin and adjust climate variables by lapse rate/barometric law
for i in range(len(climateds_binned['bin_elev'].values)):
    z = climateds_binned['bin_elev'].values[i]
    temp_adj[i,:] = gcm_temp + pygem_prms.lapserate*(gcm_elev-z)
    prec_adj[i,:] = gcm_prec*kp*(1+precgrad*(gcm_elev-z))
    sp_adj[i,:] = np.power(gcm_sp*(gcm_temp + pygem_prms.lapserate*(gcm_elev-z))/gcm_temp,
                           -gravity*molarmass_air/(R_gas*pygem_prms.lapserate))
    rh_adj[i,:] = e_func(temp_adj[i,:])/e_func(gcm_dtemp)

climateds_binned = climateds_binned.assign(bin_temp = (['pt','time'],temp_adj))
climateds_binned = climateds_binned.assign(bin_prec = (['pt','time'],prec_adj))
climateds_binned = climateds_binned.assign(bin_sp = (['pt','time'],sp_adj))
climateds_binned = climateds_binned.assign(bin_rh = (['pt','time'],rh_adj))
climateds_binned = climateds_binned.assign(bin_snow = (['pt','time'],np.where(np.array(temp_adj)<(tsnow_threshold+273),1,0)))
print('!! GCM elevation seems weirdly high and is leading to strange values here')

climateds = xr.Dataset(data_vars = dict(
    dtemp = (['glacier','time'],gcm_dtemp),
    surfrad = (['glacier','time'],gcm_surfrad),
    tcc = (['glacier','time'],gcm_tcc),
    uwind = (['glacier','time'],gcm_uwind),
    vwind = (['glacier','time'],gcm_vwind),
),coords = dict(glacier=glac_no,time=gcm_hours))

!! GCM elevation seems weirdly high and is leading to strange values here


/tmp/ipykernel_3183/2635083039.py:26: RuntimeWarning: invalid value encountered in power
  sp_adj[i,:] = np.power(gcm_sp*(gcm_temp + pygem_prms.lapserate*(gcm_elev-z))/gcm_temp,


In [ ]:
class meltProfile():
    """
    Temperature and density profile function to distribute melt through vertical bins and define
    the content (snow or firn) of each bin.
    
    Attributes
    ----------
    options? : bool
        options to dictate what parameterizations are used?
    """ 
    def __init__(self):
        """
        Add variable name and specific properties associated with each gcm.
        """
        

In [13]:
melt_monthly = []
time_idx = 0 #initiate time index to scale with loop
# treat as constants now, add parameterizations later:
# these are randomly thrown in values they don't mean anything
albedo = .5 
vp = 1 #vapor pressure of water: depends on the temperature
cpw = 1 #heat capacity of water
cpa = 1 #heat capacity of air

# ===== ENTER HOURLY LOOP!! =====
for hour in gcm_hours[0:10]:
    if time_idx<1:
        hourly_Q = []
        pass # need to calculate monthly_Q before can append to melt array
    elif hour.is_month_start and hour.hour > 1:
        # convert previous month Q to M, summing hourly Q_melts
        monthly_M = np.sum(hourly_Q)*3600/(density_water*Lh_rf)
        melt_monthly.append(monthly_M)
        hourly_Q = []

    #later this should become where we start the glacier loop
    # for glacier in ran(len(glac_no)):

    result = solar.get_position(hour,glacier_table['CenLon'],glacier_table['CenLat'])
    Snet_surf = climateds['surfrad'][time_idx] * (1-albedo) #* (cos(theta))
    #Snet = Snet_surf*vert_bins['lambdas']*np.exp(-vert_bins['bin_z']*vert_bins['rs'])
        
    #parameterization for Lnet from COSIPY
    Ecs = .23+ .433*(vp/(gcm_temp[i][time_idx]+273.15))**(1/8)
    Lnet = Ecs*(1-climateds['tcc'][time_idx]**2)+.984*gcm_tcc[i][time_idx]**2

    w10 = np.sqrt(climateds['uwind'][time_idx]**2+climateds['vwind'][time_idx]**2)
    rain_mask = -(climateds_binned['bin_snow']-1)
    Qp = rain_mask*cpw*(climateds_binned['bin_temp'][time_idx]-Ts)*climateds_binned['bin+prec'][time_idx]

    k0 = 0.4
    zm = 10
    zv = zh = 2
    if snow:
        z0m = 0.001
        z0v= 0.001
            z0h = 0.001
    elif ice:
        z0m = 0.016
        z0v = 0.004
        z0v = 0.004
    kH = k0**2/(np.log(zm/z0m)*np.log(zv/z0v))
    kE = k0**2/(np.log(zm/z0m)*np.log(zh/z0h))
    density_air = gcm_sp[i][time_idx]/R_gas/gcm_temp[i][time_idx]
    Qs = density_air*gcm_sp[0][time_idx]*cpa*kH*w10*(gcm_temp[i][time_idx]-Ts)/pressure_std
    Ql = 0.622*density_air*kE*w10*Lv*()

    Qm = Snet_surf + Lnet + Qp
    hourly_Q.append(Qm)

    time_idx +=1


<xarray.DataArray (pt: 3, vert_idx: 10)>
array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Coordinates:
    point     (pt) <U6 'Bottom' 'Middle' 'Top'
  * vert_idx  (vert_idx) int64 0 1 2 3 4 5 6 7 8 9
Dimensions without coordinates: pt
<xarray.DataArray (pt: 3, vert_idx: 10)>
array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Coordinates:
    point     (pt) <U6 'Bottom' 'Middle' 'Top'
  * vert_idx  (vert_idx) int64 0 1 2 3 4 5 6 7 8 9
Dimensions without coordinates: pt
<xarray.DataArray (pt: 3, vert_idx: 10)>
array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Coordinates:
    point     (pt) <U6 'Bottom' 'Middle' 'Top'
  * vert_idx  (vert_idx) int64 0 1 2 3 4 5 6 7 8 9
Dimensions without coordinates

In [ ]:
import pygem.pygem_input as pygem_prms
import pickle

In [ ]:
glacier_str = glac_no[0][1::]
modelprms_fn =  glacier_str + '-modelprms_dict.pkl'
modelprms_fp = pygem_prms.output_filepath + 'calibration/' + glacier_str.split('.')[0].zfill(2) + '/'
modelprms_fullfn = modelprms_fp + modelprms_fn
assert os.path.exists(modelprms_fullfn), glacier_str + ' calibrated parameters do not exist.'            
with open(modelprms_fullfn, 'rb') as f:
    modelprms_dict = pickle.load(f)